In [1]:
import torch 
import torch.nn as nn

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0)) 

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
train_df_X = torch.tensor([ [0., 0.],
                            [0., 1.],
                            [1., 0.],
                            [1., 1.] ], device = device)
train_df_Y = torch.tensor([ [0.],
                            [1.],
                            [1.],
                            [0.] ], device = device)

In [4]:
class XORNN(nn.Module):
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.hidden = nn.Linear(2, 2)
        self.relu = nn.ReLU()
        self.output = nn.Linear(2, 1)

    def forward(self, x):
        z1 = self.hidden(x)
        x1 = self.relu(z1)
        z2 = self.output(x1)
        x2 = z2
        return x2

In [5]:
model = XORNN()
model.to(device)
model

XORNN(
  (hidden): Linear(in_features=2, out_features=2, bias=True)
  (relu): ReLU()
  (output): Linear(in_features=2, out_features=1, bias=True)
)

In [6]:
loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.Adam(model.parameters(), lr = 0.01)

In [7]:
for epoch in range(5000):
    logits = model(train_df_X)
    loss = loss_fn(logits, train_df_Y)
    optim.zero_grad()
    loss.backward()
    optim.step()
    if epoch % 500 == 0 : print(f"Epoch: {epoch}, Loss: {loss.item()}")


Epoch: 0, Loss: 0.6641425490379333
Epoch: 500, Loss: 0.016411500051617622
Epoch: 1000, Loss: 0.0042417095974087715
Epoch: 1500, Loss: 0.0019091974245384336
Epoch: 2000, Loss: 0.0010592897888273
Epoch: 2500, Loss: 0.000655105453915894
Epoch: 3000, Loss: 0.00043223638203926384
Epoch: 3500, Loss: 0.00029727519722655416
Epoch: 4000, Loss: 0.0002104002342093736
Epoch: 4500, Loss: 0.00015237246407195926


In [8]:
print(model.hidden.weight)

Parameter containing:
tensor([[ 3.3643, -3.3645],
        [-3.3544,  3.3545]], device='cuda:0', requires_grad=True)


In [9]:
print(model.hidden.bias)

Parameter containing:
tensor([ 5.0069e-05, -1.6153e-04], device='cuda:0', requires_grad=True)


In [10]:
print(model.output.weight)

Parameter containing:
tensor([[5.5130, 5.5289]], device='cuda:0', requires_grad=True)


In [11]:
print(model.output.bias)

Parameter containing:
tensor([-8.6657], device='cuda:0', requires_grad=True)


In [12]:
torch.save(model.state_dict(), "XOR_model.pth")

In [13]:
torch.cuda.empty_cache()